# Cornucopia: circuit-distance upper bounds

Search for a nontrivial logical fault with zero detector syndrome. Each trial
adds a random logical constraint and solves the resulting syndrome problem
with BP-OSD. The minimum valid fault weight found is an upper bound on circuit
distance. This workflow uses CX faults only; measurement and reset flips are
zero when constructing the detector error model.


## Parameters


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "code_construction" / "affine_codes.py").is_file()
)
sys.path.insert(0, str(REPO_ROOT))

In [ ]:
import os
import shlex
import subprocess
import sys
from pathlib import Path

PYTHON = Path(sys.executable)
SCRIPT = REPO_ROOT / "circuit_distance/estimate_distance.py"
OUTPUT_DIR = Path("circuit_distance/results")

CODES = (
    "cornucopia_p147_d14"  # Codes to run: "all" or comma-separated names, e.g. "cornucopia_p87_d12"
)
BASIS = "Z"  # Memory/readout basis: "Z", "X", or "both"
NUM_TRIALS = 20000  # Total BP-OSD random logical-constraint trials per code and basis
WORKERS = max(1, min(8, os.cpu_count() or 1))
CHUNK_SIZE = 100  # Number of trials assigned to each worker task chunk
P_CX = 0.003  # Two-qubit gate depolarizing probability used to build the DEM priors
SEED = 20260702  # Seed for logical-constraint sampling
OSD_ORDER = 3  # OSD search order passed to BP-OSD
MAX_ITER = 1000  # Maximum BP iterations inside each BP-OSD solve
PROGRESS_SECONDS = 30  # Minimum seconds between progress prints unless a new best is found
LOGICAL_COMBO_SIZE = 0  # 0 samples any nonzero logical combination; >0 fixes that combination size

assert PYTHON.exists(), PYTHON
(REPO_ROOT / OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print(f"python={PYTHON.name}")
print(f"script={SCRIPT.relative_to(REPO_ROOT)}")
print(f"workers={WORKERS}")
print(f"output={OUTPUT_DIR}")

## Command


In [ ]:
cmd = [
    str(PYTHON),
    str(SCRIPT),
    "--codes",
    CODES,
    "--basis",
    BASIS,
    "--num-trials",
    str(NUM_TRIALS),
    "--workers",
    str(WORKERS),
    "--chunk-size",
    str(CHUNK_SIZE),
    "--p-cx",
    str(P_CX),
    "--seed",
    str(SEED),
    "--osd-order",
    str(OSD_ORDER),
    "--max-iter",
    str(MAX_ITER),
    "--logical-combo-size",
    str(LOGICAL_COMBO_SIZE),
    "--progress-seconds",
    str(PROGRESS_SECONDS),
    "--output-dir",
    str(OUTPUT_DIR),
]
print(" ".join(shlex.quote(part) for part in ["python", *cmd[1:]]))

## Run


In [ ]:
process = subprocess.Popen(
    cmd,
    cwd=str(REPO_ROOT),
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end="")
return_code = process.wait()
if return_code:
    raise SystemExit(return_code)

## Results


In [ ]:
summary = REPO_ROOT / OUTPUT_DIR / "summary.txt"
if summary.exists():
    print(summary.read_text())
else:
    print(f"No summary yet: {summary}")